In [ ]:
from typing import Dict, Tuple
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models, transforms
from torchvision.datasets import MNIST
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import numpy as np

%matplotlib inline

from torch import autograd
from torch.autograd import Variable
from tensorboardX import SummaryWriter
import torch.optim as optim
import torchvision.datasets as datasets
import time
import os

if __name__ == "__main__":
    print("Torch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA version:", torch.version.cuda)
    print("Number of GPUs:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.device_count() > 0 else "No GPU detected")

In [ ]:
import importlib
import waveguide_dataset_paired_nonorm
importlib.reload(waveguide_dataset_paired_nonorm)
from waveguide_dataset_paired_nonorm import WaveguideDatasetPaired

h5_path='train_test_split.h5'
stats_path="waveguide_stats_log_norm_above90.npz"

train_ds = WaveguideDatasetPaired(h5_path, split="train", stats_path=stats_path)
test_ds  = WaveguideDatasetPaired(h5_path, split="test",  stats_path=stats_path)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=True, num_workers=2, pin_memory=True)

In [ ]:
class CombinedModeWeightNet(nn.Module):
    def __init__(self, mode_model, weight_model):
        super().__init__()
        self.mode_model = mode_model
        self.weight_model = weight_model

    def forward(self, x_img, x_cond):
        mode = self.mode_model(x_img, x_cond)   # [B, 1]
        weight = self.weight_model(x_img, x_cond)  # [B, 1]
        return torch.cat((mode, weight), dim=1)   # [B, 2]

from only_mode_only_weight_v3 import No_normal_modewieght_net

device = 'cuda' if torch.cuda.is_available() else 'cpu'

mode0_model = No_normal_modewieght_net().to(device)
weight0_model = No_normal_modewieght_net().to(device)

mode0_model.load_state_dict(torch.load('models/only_first_mode_no_normalization_with_less_dropout.pth', map_location=torch.device(device)))
weight0_model.load_state_dict(torch.load('models/only_first_weight_no_normalization_with_less_dropout_100.pth', map_location=torch.device(device)))

cnn = CombinedModeWeightNet(mode0_model, weight0_model).to(device)
cnn.eval()

In [ ]:
import os
import h5py
import numpy as np
from typing import Iterable, Tuple, Optional
from tqdm import tqdm
import torch

# ------------------------- utilities -------------------------

def copy_attrs(src_obj, dst_obj):
    for k, v in src_obj.attrs.items():
        dst_obj.attrs[k] = v

def iter_datasets(grp: h5py.Group, prefix="") -> Iterable[Tuple[str, h5py.Dataset]]:
    for k, v in grp.items():
        full = f"{prefix}/{k}" if prefix else k
        if isinstance(v, h5py.Group):
            yield from iter_datasets(v, full)
        elif isinstance(v, h5py.Dataset):
            yield full, v

def create_like(out_group, name, shape, dtype, src: h5py.Dataset):
    """Create a dataset in out_group named `name` with same filters/chunking as src."""
    return out_group.create_dataset(
        name,
        shape=shape,
        dtype=dtype,
        chunks=src.chunks,
        compression=src.compression,
        compression_opts=src.compression_opts,
        shuffle=getattr(src, "shuffle", None),
        fletcher32=getattr(src, "fletcher32", False),
        scaleoffset=getattr(src, "scaleoffset", None),
    )

def ensure_nchw(x: np.ndarray) -> np.ndarray:
    """
    Accept [N,H,W] or [N,1,H,W] or [N,C,H,W]; return [N,1,H,W] (pick C=0 if C>1).
    (No value scaling—just shape handling.)
    """
    if x.ndim == 3:
        N, H, W = x.shape
        return x.reshape(N, 1, H, W)
    elif x.ndim == 4:
        if x.shape[1] == 1:
            return x
        # pick channel 0 if multi-channel
        return x[:, :1, ...]
    else:
        raise ValueError(f"Unsupported pattern shape {x.shape}; expected [N,H,W] or [N,1,H,W] or [N,C,H,W].")

def extract_mode_weight(y: torch.Tensor):
    """
    CNN output handler:
      - If last dim == 2 → [mode, weight]
      - If last dim >= 5 → use indices 0 (mode0) and 4 (weight0)
    """
    D = y.shape[-1]
    if D == 2:
        return y[..., 0], y[..., 1]
    if D >= 5:
        return y[..., 0], y[..., 4]
    raise ValueError(f"CNN output last-dim {D} not supported (expect 2 or ≥5).")

# ------------------------- main routine -------------------------

def make_predicted_h5_with_param_overrides(
    src_path: str,
    dst_path: str,
    cnn,                          # your loaded model
    device: str = "cuda",
    batch_size: int = 1024,
    # dataset names (override if your file uses different ones)
    pattern_train_key: str = "pattern_train",
    pattern_test_key:  str = "pattern_test",
    params_train_key:  str = "params_train",
    params_test_key:   str = "params_test",
    # where to store predictions
    mode_train_key:    str = "mode_pred_train",
    weight_train_key:  str = "weight_pred_train",
    mode_test_key:     str = "mode_pred_test",
    weight_test_key:   str = "weight_pred_test",
    # param overrides (0-based indices 2 and 3)
    idx2_value: float = 3.45,
    idx3_value: float = 1.46,
):
    """
    Create a new H5 file identical to src, except:
      - params_* datasets have indices 2 and 3 set to the provided constants
      - new datasets with CNN predictions are added:
           mode_pred_{train,test}, weight_pred_{train,test}
    """
    assert os.path.exists(src_path), f"not found: {src_path}"

    cnn.eval()
    device = torch.device(device)

    with h5py.File(src_path, "r") as fin, h5py.File(dst_path, "w") as fout:
        # Copy file-level attrs
        copy_attrs(fin, fout)

        # First mirror the group structure and copy all datasets as-is
        # We'll overwrite/replace params_* and add prediction datasets afterward.
        def mirror_groups(src_grp: h5py.Group, dst_grp: h5py.Group):
            copy_attrs(src_grp, dst_grp)
            for name, item in src_grp.items():
                if isinstance(item, h5py.Group):
                    new_grp = dst_grp.create_group(name)
                    mirror_groups(item, new_grp)
                elif isinstance(item, h5py.Dataset):
                    src_ds = item
                    dst_ds = create_like(dst_grp, name, src_ds.shape, src_ds.dtype, src_ds)
                    copy_attrs(src_ds, dst_ds)
                    # byte-for-byte copy of data
                    dst_ds[...] = src_ds[...]
                else:
                    raise RuntimeError(f"Unknown HDF5 object at {src_grp.name}/{name}")

        mirror_groups(fin, fout)

        # Helper to get (pattern_dset, params_dset) by name, raising if missing
        def fetch_pair(pattern_key: str, params_key: str) -> Tuple[h5py.Dataset, h5py.Dataset]:
            if pattern_key not in fin or params_key not in fin:
                # Try to search in case they are nested
                avail = dict(iter_datasets(fin))
                patt = [k for k in avail if k.endswith(pattern_key) or k.split("/")[-1] == pattern_key]
                par  = [k for k in avail if k.endswith(params_key)  or k.split("/")[-1] == params_key]
                if not patt or not par:
                    raise KeyError(f"Could not locate datasets '{pattern_key}' and/or '{params_key}' in {src_path}")
                p_key = patt[0]; q_key = par[0]
                print(f"[info] resolved '{pattern_key}' -> '{p_key}', '{params_key}' -> '{q_key}'")
                return fin[p_key], fin[q_key]
            return fin[pattern_key], fin[params_key]

        # Process both splits
        for split_name, patt_key, par_key, m_key, w_key in [
            ("train", pattern_train_key, params_train_key, mode_train_key,  weight_train_key),
            ("test",  pattern_test_key,  params_test_key,  mode_test_key,   weight_test_key),
        ]:
            print(f"\nProcessing split: {split_name}")
            patt_src, par_src = fetch_pair(patt_key, par_key)
            patt_dst = fout[patt_src.name]  # already copied 1:1
            par_dst  = fout[par_src.name]

            # Sanity checks
            if par_src.shape[-1] < 4:
                raise ValueError(f"{par_src.name} last dimension < 4; cannot set indices 2 and 3.")
            # Write overridden params into the *destination* dataset
            # (We stream in chunks to avoid loading everything if large.)
            N = par_src.shape[0] if par_src.ndim >= 2 else par_src.shape[0]
            par_dtype = par_src.dtype

            # Create prediction datasets mirroring params' filter settings
            mode_ds = create_like(par_dst.parent, m_key, (N,), np.float32, par_dst)
            weight_ds = create_like(par_dst.parent, w_key, (N,), np.float32, par_dst)
            mode_ds.attrs["generated_by"] = "cnn"
            weight_ds.attrs["generated_by"] = "cnn"
            mode_ds.attrs["params_override_idxs_2_3"] = True
            weight_ds.attrs["params_override_idxs_2_3"] = True
            mode_ds.attrs["idx2_value"] = float(idx2_value)
            mode_ds.attrs["idx3_value"] = float(idx3_value)
            weight_ds.attrs["idx2_value"] = float(idx2_value)
            weight_ds.attrs["idx3_value"] = float(idx3_value)

            # Iterate in batches: read patterns + params, override params, run CNN, write back
            # Try to align batch size with chunks
            bs = max(1, batch_size)
            total = N
            with tqdm(total=total, desc=f"Predicting {split_name}", unit="item") as pbar:
                for start in range(0, N, bs):
                    end = min(start + bs, N)

                    # Read patterns (support [N,H,W] or [N,1,H,W] or [N,C,H,W])
                    patt_np = patt_src[start:end, ...]
                    patt_np = np.asarray(patt_np)  # actual numpy array
                    patt_nchw = ensure_nchw(patt_np)  # [B,1,H,W]
                    B, _, H, W = patt_nchw.shape

                    # Read params and override indices 2 and 3
                    par_np = np.asarray(par_src[start:end, ...])  # [B,4...]
                    par_np = par_np.copy()  # we'll edit in place
                    par_np[..., 2] = idx2_value
                    par_np[..., 3] = idx3_value

                    # Write the overridden params back into the destination file
                    par_dst[start:end, ...] = par_np.astype(par_dtype, copy=False)

                    # Prepare tensors for cnn (no clamping or scaling from this script)
                    x = torch.from_numpy(patt_nchw).to(device=device, dtype=torch.float32)
                    p = torch.from_numpy(par_np).to(device=device, dtype=torch.float32)

                    with torch.no_grad():
                        y = cnn(x, p)  # [B, D]
                        mode_t, weight_t = extract_mode_weight(y)  # [B], [B]

                    # Move preds to numpy and write
                    mode_ds[start:end]   = mode_t.detach().cpu().numpy().astype(np.float32, copy=False)
                    weight_ds[start:end] = weight_t.detach().cpu().numpy().astype(np.float32, copy=False)

                    pbar.update(end - start)

        print(f"\nDone. Wrote: {dst_path}")

# ------------------------- example usage -------------------------
# Provide your loaded cnn and device. This does not alter the source file.
#
make_predicted_h5_with_param_overrides(
    src_path="train_test_split.h5",
    dst_path="fixed_material_dataset.h5",
    cnn=cnn,
    device="cuda" if torch.cuda.is_available() else "cpu",
    batch_size=1024,
    # If your dataset keys differ, set them here:
    pattern_train_key="pattern_train",
    pattern_test_key="pattern_test",
    params_train_key="params_train",
    params_test_key="params_test",
)


In [ ]:
import h5py
import numpy as np
import random
from typing import Dict, Tuple, Optional

# --------- Utilities ---------

def walk_h5(obj, prefix=""):
    """
    Recursively print the HDF5 tree with dataset shapes/dtypes.
    """
    if isinstance(obj, h5py.File):
        print(f"{obj.filename} (root)")
        for k, v in obj.items():
            walk_h5(v, k)
    elif isinstance(obj, h5py.Group):
        print(f"[GROUP] {prefix}/")
        for k, v in obj.items():
            walk_h5(v, f"{prefix}/{k}" if prefix else k)
    elif isinstance(obj, h5py.Dataset):
        print(f"  [DATASET] {prefix}  shape={obj.shape}  dtype={obj.dtype}")
    else:
        print(f"  [UNKNOWN] {prefix}")

def list_all_datasets(f: h5py.File) -> Dict[str, h5py.Dataset]:
    """
    Return dict of {full_path: dataset}.
    """
    out = {}
    def _recur(g, p=""):
        for k, v in g.items():
            full = f"{p}/{k}" if p else k
            if isinstance(v, h5py.Group):
                _recur(v, full)
            elif isinstance(v, h5py.Dataset):
                out[full] = v
    _recur(f)
    return out

def find_dataset(f: h5py.File, name: str) -> Optional[str]:
    """
    Find dataset by exact path or by basename matching `name`.
    Returns the full path or None.
    """
    if name in f:
        return name
    all_sets = list_all_datasets(f)
    # try basename match
    matches = [full for full in all_sets if full.split("/")[-1] == name]
    if matches:
        return matches[0]
    return None

def ensure_hw(slice_like: np.ndarray) -> np.ndarray:
    """
    Accept [H,W], [1,H,W], or [C,H,W]; return [H,W] (take channel 0 if present).
    """
    if slice_like.ndim == 2:
        return slice_like
    if slice_like.ndim == 3:
        return slice_like[0]  # first channel
    raise ValueError(f"Unexpected array rank for pattern slice: {slice_like.shape}")

def get_length_from_pattern(d: h5py.Dataset) -> int:
    """
    Infer number of items N from a pattern dataset with shape [N,H,W] or [N,1,H,W] or [N,C,H,W].
    If dataset is 2D, treat N=1.
    """
    if d.ndim == 2:
        return 1
    return d.shape[0]

def get_item_pattern(d: h5py.Dataset, idx: int) -> np.ndarray:
    """
    Return a 2D array [H,W] for item idx from pattern dataset.
    Supports [N,H,W], [N,1,H,W], [N,C,H,W].
    """
    if d.ndim == 2:
        return ensure_hw(d[...])
    if d.ndim == 3:
        return ensure_hw(d[idx, ...])
    if d.ndim == 4:
        return ensure_hw(d[idx, ...])
    raise ValueError(f"Unsupported pattern dataset rank: {d.ndim}")

def get_item_params(d: h5py.Dataset, idx: int) -> np.ndarray:
    """
    Return params row for item idx. Supports [N, D] or [N, ...].
    """
    if d.ndim >= 2:
        return np.array(d[idx, ...])
    raise ValueError(f"Unsupported params dataset rank: {d.ndim}")

def get_item_scalar(d: h5py.Dataset, idx: int) -> float:
    """
    Return a scalar prediction value for item idx from a 1D dataset of length N.
    """
    arr = d[idx]
    return float(np.array(arr))

def pretty_print_item(split: str,
                      idx: int,
                      pattern: np.ndarray,
                      params: np.ndarray,
                      mode_pred: Optional[float],
                      weight_pred: Optional[float]):
    """
    Print all values for one item (pattern matrix, params vector, and labels).
    """
    print(f"\n========== {split.upper()} ITEM @ index {idx} ==========")
    print("Params (full vector):")
    with np.printoptions(suppress=False, linewidth=120, threshold=np.inf):
        print(params)
    print("\nPattern [H,W] matrix (all values):")
    with np.printoptions(suppress=False, linewidth=120, threshold=np.inf):
        print(pattern)
    if mode_pred is not None:
        print(f"\nLabel (mode_pred_{split}):   {mode_pred}")
    else:
        print(f"\nLabel (mode_pred_{split}):   <not found>")
    if weight_pred is not None:
        print(f"Label (weight_pred_{split}): {weight_pred}")
    else:
        print(f"Label (weight_pred_{split}): <not found>")

# --------- Main inspection function ---------

def inspect_new_h5(
    h5_path: str,
    pattern_train_key: str = "pattern_train",
    pattern_test_key:  str = "pattern_test",
    params_train_key:  str = "params_train",
    params_test_key:   str = "params_test",
    mode_train_key:    str = "mode_pred_train",
    weight_train_key:  str = "weight_pred_train",
    mode_test_key:     str = "mode_pred_test",
    weight_test_key:   str = "weight_pred_test",
    seed: Optional[int] = None,
):
    """
    - Prints the tree of the H5 file with dataset names/shapes/dtypes.
    - Randomly selects one item from TRAIN and one from TEST, and prints:
        * the full params vector
        * the full pattern matrix values
        * the predicted top mode and weight (if present)
    """
    rng = random.Random(seed)

    with h5py.File(h5_path, "r") as f:
        print("\n=== H5 CONTENTS ===")
        walk_h5(f)

        # Resolve dataset paths (works even if they are nested)
        def res_or_die(name: str) -> str:
            path = find_dataset(f, name)
            if path is None:
                raise KeyError(f"Dataset '{name}' not found anywhere in {h5_path}")
            return path

        patt_train_path  = res_or_die(pattern_train_key)
        patt_test_path   = res_or_die(pattern_test_key)
        params_train_path = res_or_die(params_train_key)
        params_test_path  = res_or_die(params_test_key)

        # Predictions are optional (but expected in your new file)
        mode_train_path   = find_dataset(f, mode_train_key)
        weight_train_path = find_dataset(f, weight_train_key)
        mode_test_path    = find_dataset(f, mode_test_key)
        weight_test_path  = find_dataset(f, weight_test_key)

        patt_train = f[patt_train_path]
        patt_test  = f[patt_test_path]
        par_train  = f[params_train_path]
        par_test   = f[params_test_path]

        N_train = get_length_from_pattern(patt_train)
        N_test  = get_length_from_pattern(patt_test)
        idx_train = rng.randrange(N_train)
        idx_test  = rng.randrange(N_test)

        # TRAIN item
        pattern_tr = get_item_pattern(patt_train, idx_train)
        params_tr  = get_item_params(par_train, idx_train)

        mode_tr = get_item_scalar(f[mode_train_path], idx_train) if mode_train_path else None
        weight_tr = get_item_scalar(f[weight_train_path], idx_train) if weight_train_path else None

        pretty_print_item("train", idx_train, pattern_tr, params_tr, mode_tr, weight_tr)

        # TEST item
        pattern_te = get_item_pattern(patt_test, idx_test)
        params_te  = get_item_params(par_test, idx_test)

        mode_te = get_item_scalar(f[mode_test_path], idx_test) if mode_test_path else None
        weight_te = get_item_scalar(f[weight_test_path], idx_test) if weight_test_path else None

        pretty_print_item("test", idx_test, pattern_te, params_te, mode_te, weight_te)

# --------------- Example usage ---------------
# inspect_new_h5("train_test_split_with_preds.h5", seed=42)
